In [1]:
import pandas as pd
import numpy as np
import cornac

DATA_PATH = '../data'

# Load ratings - this is all we need for collaborative filtering
ratings = pd.read_csv(f'{DATA_PATH}/ratings.csv')

print(ratings.shape)
print(ratings.head())

(25000095, 4)
   userId  movieId  rating   timestamp
0       1      296     5.0  1147880044
1       1      306     3.5  1147868817
2       1      307     5.0  1147868828
3       1      665     5.0  1147878820
4       1      899     3.5  1147868510


In [2]:
# Cornac needs data in (user, item, rating) tuple format
# Also needs users and items mapped to consecutive integers

from cornac.data import Dataset

# Sample for faster training, 20% of ratings
ratings_sample = ratings.sample(frac=0.2, random_state=42)

# Build Cornac dataset
data = list(zip(
    ratings_sample['userId'].astype(str),
    ratings_sample['movieId'].astype(str),
    ratings_sample['rating']
))

print(f"Total interactions: {len(data)}")
print(f"Sample: {data[:3]}")

Total interactions: 5000019
Sample: [('99476', '104374', 3.5), ('107979', '2634', 4.0), ('155372', '1614', 3.0)]


In [3]:
from cornac.eval_methods import RatioSplit
from cornac.models import SVD
from cornac.metrics import RMSE, Precision, Recall

# Split into train/test
rs = RatioSplit(
    data=data,
    test_size=0.2,
    rating_threshold=3.5,  # ratings >= 3.5 are considered "liked"
    seed=42,
    verbose=True
)

# Train SVD model
svd = SVD(
    k=50,          # 50 latent factors
    max_iter=20,   # 20 training iterations
    learning_rate=0.01,
    lambda_reg=0.02,
    seed=42,
    verbose=True
)

# Evaluate
cornac.Experiment(
    eval_method=rs,
    models=[svd],
    metrics=[RMSE(), Precision(k=10), Recall(k=10)],
    user_based=True
).run()

rating_threshold = 3.5
exclude_unknowns = True
---
Training data:
Number of users = 161940
Number of items = 36910
Number of ratings = 4000015
Max rating = 5.0
Min rating = 0.5
Global mean = 3.5
---
Test data:
Number of users = 161940
Number of items = 36910
Number of ratings = 996410
Number of unknown users = 0
Number of unknown items = 0
---
Total users = 161940
Total items = 36910

[SVD] Training started!


  0%|          | 0/20 [00:00<?, ?it/s]

Optimization finished!

[SVD] Evaluation started!


Rating:   0%|          | 0/996410 [00:00<?, ?it/s]

Ranking:   0%|          | 0/141711 [00:00<?, ?it/s]


TEST:
...
    |   RMSE | Precision@10 | Recall@10 | Train (s) | Test (s)
--- + ------ + ------------ + --------- + --------- + --------
SVD | 0.7793 |       0.0043 |    0.0078 |    2.4967 |  78.4512



In [4]:
# Get SVD scores for all user-movie pairs
# Extract the trained model
svd_model = svd

# Get all user and item IDs from the dataset
all_users = rs.train_set.uid_map  # user ID mapping
all_items = rs.train_set.iid_map  # item ID mapping

print(f"Total users in model: {len(all_users)}")
print(f"Total items in model: {len(all_items)}")
print("\nSample user mapping:")
print(list(all_users.items())[:5])

Total users in model: 161940
Total items in model: 36910

Sample user mapping:
[('138273', 0), ('110872', 1), ('148974', 2), ('46319', 3), ('118803', 4)]


In [5]:
# Test: get top 10 recommendations for a specific user
test_user = '1'  # userId 1 from our dataset

# Get internal user index
user_idx = all_users.get(test_user)
print(f"User {test_user} → internal index: {user_idx}")

# Score all movies for this user
user_scores = svd_model.score(user_idx)

# Get top 10 movie indices
top10_indices = user_scores.argsort()[::-1][:10]

# Map back to movie IDs
idx_to_item = {v: k for k, v in all_items.items()}
top10_movies = [idx_to_item[i] for i in top10_indices]
top10_scores = user_scores[top10_indices]

print(f"\nTop 10 recommendations for user {test_user}:")
for movie_id, score in zip(top10_movies, top10_scores):
    print(f"  MovieId: {movie_id} → SVD score: {score:.4f}")

User 1 → internal index: 136877

Top 10 recommendations for user 1:
  MovieId: 30701 → SVD score: 4.4695
  MovieId: 171011 → SVD score: 4.4676
  MovieId: 159817 → SVD score: 4.4627
  MovieId: 116839 → SVD score: 4.4351
  MovieId: 138835 → SVD score: 4.4195
  MovieId: 100044 → SVD score: 4.4192
  MovieId: 70186 → SVD score: 4.4129
  MovieId: 42217 → SVD score: 4.3918
  MovieId: 170705 → SVD score: 4.3824
  MovieId: 8516 → SVD score: 4.3731


In [6]:
# Map movie IDs to titles
import pandas as pd

movies = pd.read_csv(f'{DATA_PATH}/movies.csv')
movies['movieId'] = movies['movieId'].astype(str)

top10_df = pd.DataFrame({
    'movieId': top10_movies,
    'svd_score': top10_scores
})

top10_df = top10_df.merge(movies, on='movieId')
print(top10_df[['title', 'genres', 'svd_score']])

                                               title  \
0  Chimes at Midnight (Campanadas a medianoche) (...   
1                             Planet Earth II (2016)   
2                                Planet Earth (2006)   
3          Gett: The Trial of Viviane Amsalem (2014)   
4                   Return to Treasure Island (1988)   
5                                Human Planet (2011)   
6  Heimat - A Chronicle of Germany (Heimat - Eine...   
7                       Late Spring (Banshun) (1949)   
8                            Band of Brothers (2001)   
9  Matter of Life and Death, A (Stairway to Heave...   

                       genres  svd_score  
0            Comedy|Drama|War   4.469533  
1                 Documentary   4.467626  
2                 Documentary   4.462714  
3                       Drama   4.435070  
4  Adventure|Animation|Comedy   4.419528  
5                 Documentary   4.419216  
6                       Drama   4.412939  
7                       Drama   4.39184

In [7]:
ratings_user1 = ratings[ratings['userId'] == 1].copy()
ratings_user1['movieId'] = ratings_user1['movieId'].astype(str)
ratings_user1 = ratings_user1.merge(movies, on='movieId')
ratings_user1 = ratings_user1.sort_values('rating', ascending=False)
print(ratings_user1[['title', 'genres', 'rating']].head(10))

                                                title  \
0                                 Pulp Fiction (1994)   
18  Saragossa Manuscript, The (Rekopis znaleziony ...   
57                                       Dolls (2002)   
56                              Dolce Vita, La (1960)   
48       Eternal Sunshine of the Spotless Mind (2004)   
41                         Lost in Translation (2003)   
37                City of God (Cidade de Deus) (2002)   
33                            Teddy Bear (Mis) (1981)   
26                      Night, The (Notte, La) (1960)   
24      In the Mood For Love (Fa yeung nin wa) (2000)   

                                   genres  rating  
0             Comedy|Crime|Drama|Thriller     5.0  
18                Adventure|Drama|Mystery     5.0  
57                          Drama|Romance     5.0  
56                                  Drama     5.0  
48                   Drama|Romance|Sci-Fi     5.0  
41                   Comedy|Drama|Romance     5.0  
37  Acti

In [8]:
import pickle

with open(f'{DATA_PATH}/svd_model.pkl', 'wb') as f:
    pickle.dump(svd, f)

print("SVD model saved!")

SVD model saved!
